# **Section 3:** ArcGIS API for Python

**Goal for this section:** Build an understanding of the ArcGIS API for Python and the patterns we'll use

## 3.1 Introducing the ArcGIS API for Python

The **ArcGIS API for Python** is a Python library for working with **ArcGIS Online and ArcGIS Enterprise**. It allows you to interact with your organization programmatically rather than performing every task manually.

<br>

The API can be used to:

- **Query and manage content** — items, maps, layers, applications, and files
- **Work with users and groups** — search users, inspect accounts, manage groups and memberships
- **Perform GIS analysis** — access spatial analysis tools and services *(Beware of the credits!)*
- **Work with spatial data** — read, edit, publish, and manage feature layers and other GIS data
- **Create maps and visualizations** — build and work with web maps and other mapping content
- **Automate workflows** — turn repetitive GIS administration and data management tasks into Python scripts

<br>

### A simple mental model

Think of the API as a way to interact with the major components of your ArcGIS organization using Python:

>**GIS → Users | Groups | Items | Data | Maps | Analysis**

For this workshop, we'll focus primarily on **items and users**. We'll use Python to search for these objects, inspect their properties, and identify things that may need attention or review.

<br>

### Why use the API?

The API becomes particularly useful when you need to work with **large numbers of objects or repeat a task consistently**. Instead of manually checking hundreds of users or items, we can write a query or script that applies the same criteria across the entire organization.

This gives us a basic workflow we'll use throughout the workshop:

>**Query → Inspect → Filter → Flag**

<br>


### ArcGIS API for Python vs. ArcPy

Anyone interested in ArcGIS + Python has heard of **ArcPy**, so you might be wondering how it relates to the ArcGIS API for Python. Well, it doesn't! They overlap in some areas, but are seperate products and designed for different types of GIS workflows:

| | ArcPy | ArcGIS API for Python |
|---|---|---|
| **Lives inside** | ArcGIS Pro (requires a Pro license + install) | Any Python environment — Colab, a laptop, a server, anywhere |
| **Talks to** | Local data, your Pro project, ArcGIS Server geoprocessing | Your **Web GIS** — ArcGIS Online or Enterprise, over the network |
| **Best for** | Heavy geoprocessing and analysis, editing local data, map/layout automation | Portal administration, content management, sharing/publishing, users & groups, lightweight analysis via hosted tools |
| **What we're using it for today** | *Absolutely nothing* | Auditing content, flagging stale/public items, checking inactive users — administering the *organization*, not the data itself |

<br>

>**Rule of thumb:** Inside ArcGIS Pro doing geoprocessing or editing local data → reach for ArcPy. Scripting against your organization's portal, including from outside Pro entirely (like right now, in Colab) → reach for the ArcGIS API for Python.

In practice, many workflows will use **both** → ArcPy to prep and publish data locally and then the API for Python to manage and share it on the portal afterward.

If you want the longer version, here is Esri's take: [ArcPy and the ArcGIS API for Python](https://pro.arcgis.com/en/pro-app/latest/arcpy/get-started/arcpy-and-the-arcgis-api-for-python.htm).

<br>

### **Mini-lab:** Find it in the docs

Before we write more code, let's build a habit that outlasts this workshop: **knowing how to find something in the API reference yourself**, rather than memorizing method names.

<br>

Open the [ArcGIS API for Python reference](https://developers.arcgis.com/python/latest/api-reference/) in a new tab — it's organized by module (`arcgis.gis`, `arcgis.features`, `arcgis.geocoding`, etc.) — and answer the three questions below. Each has a hint if you get stuck, and a check cell underneath so you can confirm what you found before we use it.

1. **Ownership transfer.** An administrator wants to reassign a single item to a different owner (e.g. someone left the department). Find the method on the `Item` class that does this — what's it called?
2. **Filtering search results by type.** We've been calling `gis.content.search(query=...)`. What parameter would you add to only return items of a specific type, like `"Web Map"` or `"StoryMap"`?
3. **Finding a user's groups.** Is there a property on the `User` class that tells you which groups a given user belongs to?

<br>

<details>
<summary>Hints (click if stuck)</summary>

1. In `arcgis.gis`, look at the `Item` class and search the page for "reassign" or "owner".
2. Look at the `gis.content.search()` signature itself, or the "Accessing and creating content" guide page.
3. Look at the `User` class properties in `arcgis.gis` — this one isn't a method you call, just something you read.

</details>

<br>

<details>
<summary>Answers</summary>

1. `item.reassign_to(target_owner, target_folder=None)` — pass the username (or `User` object) to reassign to.
2. `item_type` — e.g. `gis.content.search(query="", item_type="Web Map")`.
3. `user.groups` — a list of `Group` objects the user belongs to.

We'll actually use #2 and #3 in the next section.
</details>

<br>

### ArcGIS API Documentation & Resources

- [ArcGIS API for Python Documentation](https://developers.arcgis.com/python/latest/)

- [API Reference](https://developers.arcgis.com/python/latest/api-reference/)

- [The GIS Module](https://developers.arcgis.com/python/latest/guide/the-gis-module/)

- [Sample Notebooks](https://developers.arcgis.com/python/latest/samples/)

- [ArcGIS REST API Documentation](https://developers.arcgis.com/rest/)

<br>

## 3.2 Connect to your Organization

In **Google Colab** (or anywhere outside of an ArcGIS environment), you need to enter your organization's URL and credentials (*never hard-code a password in a script you'll share. We'll use `getpass` to avoid this today*). Your connection string will look like this:

>```
>gis = GIS(url, username, password)`
>```

<br>

**Inside an ArcGIS Notebooks environment** (ArcGIS Online / Enterprise / ArcGIS Pro), `GIS("home")` uses your active portal session automatically — no credentials needed. Connection string:

>```
>gis = GIS("home")`
>```

<br>

What about **Enterprise logins**?! The pattern requires a [client id](https://developers.arcgis.com/python/latest/guide/working-with-different-authentication-schemes/#obtaining-a-client-id), but after that it's easy:

>```
># Replace with your organization URL and your registered Client ID
>portal_url = "https://yourorg.maps.arcgis.com"
>client_id = "YOUR_OAUTH_CLIENT_ID"
>
># This triggers the interactive SSO prompt
>gis = GIS(url=portal_url, client_id=client_id)
>```

### Get connected!

`arcgis` and your `gis` connection live only in this runtime's memory — a reconnect wipes both. If a cell below errors with `NameError: name 'gis' is not defined`, run this cell once (it reinstalls `arcgis` if needed and reconnects), then keep going.

<font color="red" size="2"><i><b>Note:</b></font><font size="2"> You may see a "pip's dependency resolver" warning error after running this install again. Just ignore it.</i></font>


In [ ]:
# Check that arcgis is installed
try:
    import arcgis
except ImportError:
    %pip install -qq arcgis
    import arcgis


Let get connected!

1.   Add your organization's url
2.   Add your username

<details>
<summary>Don't have account? Use this one!</summary>

```
url = "https://gageospatial.maps.arcgis.com"
username = "gga_workshop"
password (when prompted) = "GGAWorkshop26"
```

</details>



In [ ]:
from arcgis.gis import GIS

# Connecting to your org
import getpass
url = "https://YOURORG.maps.arcgis.com"
username = "USERNAME"
password = getpass.getpass("Password: ")
gis = GIS(url, username, password)


# Connecting inside ArcGIS Notebooks (no credentials needed)
# gis = GIS("home")


# Connecting with an Enterprise Login
# Uncomment below & replace with your organization URL and your registered Client ID
# portal_url = "https://yourorg.maps.arcgis.com"
# client_id = "YOUR_OAUTH_CLIENT_ID"
# gis = GIS(url=portal_url, client_id=client_id)



In [ ]:
# Inspect the connection
gis.properties.name


In [ ]:
# Who am I connected as?
me = gis.users.me
print(me.username)
print(me.role)
print(me.email)


**A note on credentials:** Don't hardcode your password into your notebooks or scripts. We use `getpass` here to prompt for the password, but there are [options for scripts as well](https://developers.arcgis.com/python/latest/guide/working-with-different-authentication-schemes/#storing-your-credentials-locally)

## 3.3 Basic Concepts

Let's explore some basic operations before jumping in the labs. We'll search for items and explore their structure.


### Search

Start where any audit starts — what's actually in here?

In [ ]:
items = gis.content.search(
    query="",
    max_items=10
)

for item in items:
    print(item.title, "-", item.type)

You can narrow a search with keywords, owners, item types, and sorting — including the `item_type` parameter you just found in the docs:

In [ ]:
# A few more interesting searches than a bare query=""
feature_layers = gis.content.search(query="", item_type="Feature Layer", max_items=10)
web_maps       = gis.content.search(query="", item_type="Web Map", max_items=10)
my_items       = gis.content.search(query=f"owner:{me.username}", max_items=10)

print(f"Feature Layers found: {len(feature_layers)}")
print(f"Web Maps found: {len(web_maps)}")
print(f"Items owned by {me.username}: {len(my_items)}")


Let's use the search function to do a server-side sort and pull back the 10 most recently modified items:

In [ ]:
# The org's most recently updated content
recent_items = gis.content.search(query="", sort_field="modified", sort_order="desc", max_items=10)

for item in recent_items:
    print(item.modified, "-", item.title, f"({item.type})")


### Items

Let's inspect one of those items closer — tags, description, and sharing level are exactly the properties Lab 2 will use to decide what gets flagged.

In [ ]:
recent_items[0].title


In [ ]:
# Inspecting a single item -- the one that came up first when we sorted by "most recently modified"
item = recent_items[0]

print("Title:      ", item.title)
print("Type:       ", item.type)
print("Owner:      ", item.owner)
print("ID:         ", item.id)
print("Modified:   ", item.modified)
print("Tags:       ", item.tags)
print("Description:", item.description)
print("Access:     ", item.access)   # sharing level: private / org / public


## 3.4 From ArcGIS Online to Pandas

This is a pattern that matters a lot today:

```
ArcGIS Online
      ↓
    Query
      ↓
    Data
      ↓
Pandas DataFrame
      ↓
Python analysis
      ↓
   Export
```

Once data is a DataFrame, everything you learned about Pandas in Notebook 01 applies directly. We can quickly analyze and export a report.

In [ ]:
# Pick a feature layer item from your org, then query it
# Manipulate the query string to target any item you'd like
layer_item = gis.content.search(query="type:Feature Layer", max_items=1)[0]
layer = layer_item.layers[0]

result = layer.query()
df = result.sdf   # .sdf = "spatially enabled dataframe"
df.head()


We just stepped through the process of searching for items, sorting by recency, exploring an item, and moving it to Pandas. Now let's try doing that in bulk..

<br>

---

<br>


# **Section 4:** Creating automated workflows

**Goal:** Turn the basic logic above into three practical scripts you can adapt for your organization.

```
Connect
 ↓
Search
 ↓
Iterate
 ↓
Inspect
 ↓
Apply rules
 ↓
Create report
```

Each lab below is self-contained — run its cells top to bottom. Standalone copies of these three scripts (as plain `.py` files) are available in the GitHub repo, ready to adapt and run outside a notebook.


### A quick note on privileges before we start the labs

The three labs below use different scopes of ArcGIS access:

- **Labs 1 & 2** (content inventory, content audit) work for **any authenticated org member** — `gis.content.search()` simply returns whatever content is shared with you (your own items, org-shared items, public items). If you're not an admin, your inventory/audit will just cover a narrower slice of the org than someone with broader visibility — the automation pattern is identical either way.
- **Lab 3** (inactive users) is the one that typically **needs Administrator privileges** (or a custom role with "view org members" rights) to list *other* users' accounts and login history. If you don't have that, the lab below automatically falls back to a **sample dataset** so you can still build and run the exact same flagging logic — see the note in that lab.


### Quick demo before you're on your own

Before you dive into the labs, let's build the flag pattern together on something tiny and non-live, so the shape of it is fresh going in.


In [ ]:
# A miniature version of what Lab 2 will do at full scale
sample_items = [
    {"title": "Old Parcels", "modified_days_ago": 900, "size_mb": 850},
    {"title": "Road Closures", "modified_days_ago": 15, "size_mb": 12},
]

for item in sample_items:
    flags = []
    if item["modified_days_ago"] > 730:
        flags.append("STALE")
    if item["size_mb"] > 500:
        flags.append("LARGE")
    print(item["title"], "->", flags if flags else "no flags")


That's the whole pattern: **compute a condition, `if` it's true append a flag, repeat for each rule.**

Labs 1–3 below are you applying that same idea to real (or sample) ArcGIS data — work at your own pace, and flag me down if you get stuck!

<br>


### **LAB 1** | GIS Content Inventory

**Problem:** If someone asked you *"What exactly is in our GIS?"*, could you answer without manually browsing the portal?

We'll build this one step at a time — search, inspect one item to confirm the shape of the data, build the full list, then export it.


#### **Step 1:** Search the organization.

An empty query returns everything shared with you across the org (see the privileges note above). For this lab we're limiting the response with `max_items=200`.

In [ ]:
all_items = gis.content.search(query="", max_items=200)
print(f"Found {len(all_items)} items")


#### **Step 2:** Look at one item first.

Before looping over all 200, confirm you know which properties you want, using the same ones we printed together in Section 3.3.

In [ ]:
sample_item = all_items[0]
print(sample_item.title, "|", sample_item.type, "|", sample_item.owner, "|", sample_item.id, "|", sample_item.modified)


#### **Step 3:** Now build the full inventory.

Loop over every item and pull out those same properties into a dictionary. Everything's filled in except one blank — `modified` follows the exact same pattern as `created` right above it, just reading a different timestamp field.

In [ ]:
import pandas as pd
from datetime import datetime

records = []
for item in all_items:
    records.append({
        "title": item.title,
        "type": item.type,
        "owner": item.owner,
        "id": item.id,
        "created": datetime.fromtimestamp(item.created / 1000),
        "modified": ___,   # fill in: same idea as "created", but using item.modified
    })

inventory_df = pd.DataFrame(records)
inventory_df.head()


**Hint:** look at the `"created"` line directly above the blank — swap `item.created` for the field this blank is asking about.

<details>
<summary>Click to reveal the solution</summary>

```python
"modified": datetime.fromtimestamp(item.modified / 1000),
```
</details>


#### **Step 4:** Export it.

This is the file your organization would actually use.

In [ ]:
inventory_df.to_csv("content_inventory.csv", index=False)
print(f"Saved {len(inventory_df)} items to content_inventory.csv")


**Reusable pattern:**

```
ArcGIS
  |
Search
  |
Python list
  |
DataFrame
  |
CSV
```

<br>

### **LAB 2** | Content Audit: Flag Large, Stale & Public Items

**Problem:** Which items might need to be reviewed?

We'll define three rules, prove they work on a single item, then apply them to everything at once:

- **STALE** — Modified more than 2 years ago
- **LARGE** — Size greater than a chosen threshold
- **PUBLIC** — Shared with everyone

<br>

#### **Step 1:** Set the thresholds and a couple of small helpers.

Nothing to fill in here — this is just plumbing to make the rules below easier to read. Run it and move on.

In [ ]:
STALE_DAYS = 730          # ~2 years
LARGE_SIZE_MB = 500       # adjust threshold for your org

def days_since(dt):
    return (datetime.now() - dt).days

def get_size_mb(item):
    try:
        return (item.size or 0) / (1024 * 1024)
    except Exception:
        return 0

def get_sharing(item):
    try:
        sharing = item.shared_with
        if sharing.get("everyone"):
            return "Everyone"
        elif sharing.get("org"):
            return "Organization"
        elif sharing.get("groups"):
            return "Groups"
        else:
            return "Private"
    except Exception:
        return "Unknown"


#### **Step 2:** Test the three rules on one item first.

`stale`/`large`/`public` are just comparisons, same concept as `days_open > 30` back in Notebook 01.

In [ ]:
item = all_items[0]
modified_dt = datetime.fromtimestamp(item.modified / 1000)
size_mb = get_size_mb(item)
sharing = get_sharing(item)

stale = ___    # fill in: True if days_since(modified_dt) is more than STALE_DAYS
large = ___    # fill in: True if size_mb is more than LARGE_SIZE_MB
public = ___   # fill in: True if sharing equals "Everyone"

print(f"stale={stale}, large={large}, public={public}")


**Hint:** each one is a single comparison using the constants from Step 1.

<details>
<summary>Click to reveal the solution</summary>

```python
stale = days_since(modified_dt) > STALE_DAYS
large = size_mb > LARGE_SIZE_MB
public = sharing == "Everyone"
```
</details>

<br>


#### **Step 3:** Turn those booleans into flags.

Same pattern as the demo you ran before Lab 1. We just want to check each condition and append a string if it's true.

In [ ]:
flags = []
___

print(flags if flags else "no flags")


**Hint:** three `if` blocks, one per condition, each appending its own flag string — exactly like the demo cell above.

<details>
<summary>Click to reveal the solution</summary>

```python
if stale:
    flags.append("STALE")
if large:
    flags.append("LARGE")
if public:
    flags.append("PUBLIC")
```
</details>

<br>


#### **Step 4:** Now run all of that for every item in the org, and build the audit report.

This cell puts together exactly what you just proved works on one item. No new blanks, just the same logic applied in a `for` loop.

In [ ]:
audit_records = []

for item in all_items:
    modified_dt = datetime.fromtimestamp(item.modified / 1000)
    size_mb = get_size_mb(item)
    sharing = get_sharing(item)

    stale = days_since(modified_dt) > STALE_DAYS
    large = size_mb > LARGE_SIZE_MB
    public = sharing == "Everyone"

    flags = []
    if stale:
        flags.append("STALE")
    if large:
        flags.append("LARGE")
    if public:
        flags.append("PUBLIC")

    audit_records.append({
        "title": item.title,
        "owner": item.owner,
        "modified": modified_dt.date(),
        "size_mb": round(size_mb, 1),
        "sharing": sharing,
        "flags": ", ".join(flags) if flags else "",
    })

audit_df = pd.DataFrame(audit_records)
audit_df[audit_df["flags"] != ""]


#### **Step 5:** Export it.

In [ ]:
audit_df.to_csv("content_audit.csv", index=False)
print(f"Flagged {len(audit_df[audit_df.flags != ''])} of {len(audit_df)} items for review")


**This is a real, actionable GIS administrative reporting workflow!** This same logic can be re-worked and applied to other scenarios to help your Org identify items for review and deletion. Keep those Orgs clean!

<br>


### **LAB 3** | Identify Inactive Users

**Problem:** Which user accounts might need attention?

We'll flag:

- **INACTIVE** — no login for X days
- **NEVER LOGGED IN** — no recorded login
- **OLD ACCOUNT** — created more than X years ago

<br>

**Privilege note:** Listing every user in an org (`gis.users.search()`) typically requires **Administrator** privileges. Step 1 below detects automatically whether you have that access; if not, it falls back to a small sample dataset from this workshop's repo. Either way you end up with `raw_records` in the same format as a Portal/AGOL response.

<br>


#### **Step 1:** Load the user data.

For Admins, the `org_users = gis.users.search(max_users=200)` line pulls the Orgs users (limited to 200 in this example) into the `org_users` variable and then we reformat it into `raw_records`. The rest of this code just handles the admin/no-amin logic, so everyone can work through the logic.

Nothing to fill in — just run it and check which data source printed at the bottom.

In [ ]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/nmholman/ubiquitous-barnacle/main"  # same repo as Notebook 01

import pandas as pd
from datetime import datetime

def days_since(dt):
    return (datetime.now() - dt).days

raw_records = []
source = "live ArcGIS org"

try:
    org_users = gis.users.search(max_users=200)
    if len(org_users) <= 1:
        # Some orgs return just yourself instead of raising an error -- treat as no access
        raise PermissionError("Only your own account is visible -- likely no admin privileges.")

    for u in org_users:
        last_login_ts = getattr(u, "lastLogin", -1)
        raw_records.append({
            "username": u.username,
            "full_name": getattr(u, "fullName", ""),
            "role": u.role,
            "user_type": getattr(u, "userType", ""),
            "created": datetime.fromtimestamp(u.created / 1000),
            "last_login": datetime.fromtimestamp(last_login_ts / 1000) if last_login_ts not in (None, -1) else None,
        })

except Exception as e:
    source = "sample dataset (no admin access detected)"
    print(f"Couldn't list org users ({e}). Falling back to sample data so you can still run this lab.\n")

    import os, urllib.request
    if not os.path.exists("sample_org_users.csv"):
        urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/data/sample_org_users.csv", "sample_org_users.csv")

    sample = pd.read_csv("sample_org_users.csv")
    for _, u in sample.iterrows():
        last_login_str = u["last_login"]
        raw_records.append({
            "username": u["username"],
            "full_name": u["full_name"],
            "role": u["role"],
            "user_type": u["user_type"],
            "created": datetime.strptime(u["created"], "%Y-%m-%d"),
            "last_login": datetime.strptime(last_login_str, "%Y-%m-%d") if isinstance(last_login_str, str) and last_login_str else None,
        })

print(f"Data source: {source}")
print(f"{len(raw_records)} user records loaded.")


#### **Step 2:** Test the two threshold rules on one record first.

`raw_records` is a plain list of dictionaries — same shape regardless of where the data came from.

In [ ]:
u = raw_records[0]

days_inactive = days_since(u["last_login"]) if u["last_login"] is not None else None
is_inactive = days_inactive is not None and days_inactive > ___    # fill in: which constant caps "too many days"? (define it here too)
is_old = days_since(u["created"]) > ___ * 365                      # fill in: which constant, in years?

print(f"days_inactive={days_inactive}, INACTIVE?={is_inactive}, OLD ACCOUNT?={is_old}")


**Hint:** these are the same two thresholds from the problem statement above — `180` days and `3` years. Define them as named constants (e.g. `INACTIVE_DAYS = 180`) rather than typing the raw numbers, since Step 3 reuses them.

<details>
<summary>Click to reveal the solution</summary>

```python
INACTIVE_DAYS = 180
OLD_ACCOUNT_YEARS = 3

is_inactive = days_inactive is not None and days_inactive > INACTIVE_DAYS
is_old = days_since(u["created"]) > OLD_ACCOUNT_YEARS * 365
```
</details>

<br>


#### **Step 3:** Apply to all records

Apply this to every user, and add the one flag we haven't handled yet: an account that has never logged in at all.

In [ ]:
INACTIVE_DAYS = 180
OLD_ACCOUNT_YEARS = 3

user_records = []
for u in raw_records:
    flags = []

    if u["last_login"] is None:
        flags.append(___)   # fill in: the flag string for "no login ever recorded"
        days_inactive = None
    else:
        days_inactive = days_since(u["last_login"])
        if days_inactive > INACTIVE_DAYS:
            flags.append("INACTIVE")

    if days_since(u["created"]) > OLD_ACCOUNT_YEARS * 365:
        flags.append("OLD ACCOUNT")

    user_records.append({
        "username": u["username"],
        "full_name": u["full_name"],
        "role": u["role"],
        "user_type": u["user_type"],
        "created": u["created"].date(),
        "days_inactive": days_inactive,
        "flags": ", ".join(flags) if flags else "",
    })

users_df = pd.DataFrame(user_records)
users_df[users_df["flags"] != ""]


**Hint:** a literal string, exactly like `"STALE"`/`"LARGE"`/`"PUBLIC"` in Lab 2.

<details>
<summary>Answer</summary>

```python
flags.append("NEVER LOGGED IN")
```
</details>

<br>


#### **Step 4:** Export it.

In [ ]:
users_df.to_csv("inactive_users.csv", index=False)
print(f"Flagged {len(users_df[users_df.flags != ''])} of {len(users_df)} users for review")


**Where to go from here?** This workflow can help with license reclamation, offboarding, seasonal employees, contractors, account cleanup, onboarding audits, service accounts.

<br>

### Admin Automation Wrap-Up

```
gis_admin/
│
├── content_inventory.py
├── content_audit.py
└── inactive_users.py
```

The same basic Python patterns power all three labs: `for` loops, `if` statements, dictionaries, lists, functions, DataFrames, CSV output — everything from Notebook 1.

These could eventually be run on a schedule, incorporated into an administrative notebook, or extended into a larger GIS governance toolkit.

---
**Take the shared workshop break.** After the break, we shift gears entirely — Python + DuckDB + modern open geospatial data.
